In [4]:
import pandas as pd 
import matplotlib.pyplot as plt 
import matplotlib
import seaborn as sns

sns.set_style("darkgrid")
matplotlib.rcParams['figure.figsize'] = (10, 6)

In [5]:
#loading the combined dataset 
df = pd.read_csv('data/raw/combined_raw.csv')

print('Shape:', df.shape)
print('Columns:', df.columns.tolist())

Shape: (25142, 7)
Columns: ['university', 'id', 'webPublicationDate', 'sectionId', 'sectionName', 'webTitle', 'wordcount']


# 1 Introduction

## 1.1 Motivation

Universities are central to public life in the UK. Debates around tuition fees, research funding, immigration, and student welfare regularly make national news. The Guardian is one of the UK's most widely read broadsheet newspapers and covers higher education widely. 

We were interested in whether Guardian coverage of universities is concentrated among a small number of institutions, and how that has changed over time, and whether major UK events produce visible spikes in coverage. These are questions we can answer using data from the Guardian API, which we learnt about in the course. 

## 1.2 Research Questions 

We have three main research questions:
1. Which Russell Group universities receive the most Guardian coverage, and how has that changed over time (2010-2024)?
2. What topics dominate coverage of the top universities, and do they differ across institutions? 
3. How does coverage spike around major UK events, such as Brexit, COVID-19, and tuition fee changes?

## 1.3 Why Russell Group Universities? Why these 10?

The Russell Group is a group of 24 research-intensive universities in the UK. They are the universities most likely to appear regularly in national press coverage, so we expected enough articles per university to make meaningful comparisons over time.
    
Rather than analysing all 24, we narrowed our focus to the top 10 by total Guardian article count between 2010 and 2024. This keeps the project manageable while still covering a range of institution types. The bar chart in data_collection.ipynb shows article counts across all Russell Group universities. There is a clear drop-off after the 10th university, which justifies our cut-off. We could not find existing work that compares Guardian coverage across Russell Group universities over a 15-year period.

## 1.4 Limitations of the search approach

We searched for each university using its full name in quotes (e.g. "University of Oxford"), which returns only articles containing the exact phrase. This means articles that use abbreviations or informal names may be missed - for example, an article mentioning "Oxford" but not "University of Oxford" would not appear in our results. Similarly, searching for "London School of Economics" will not capture articles that only refer to "LSE". We also note that "LSE" can refer to the London Stock Exchange, so using the full name avoids false matches. This means we may miss some relevant articles, but we avoid picking up irrelevant ones like articles about the London Stock Exchange.

# 2 Data Acquisition

## 2.1 The Guardian API 
We collected our data using the Guardian Open Platform API (https://open-platform.theguardian.com/documentation/). This is a free API that gives structured access to Guardian articles. To use it, we registered for a developer API key at https://open-platform.theguardian.com/access/ and stored the key in a keys.json file. 

The API returns data in JSON format. Each response contains a response object with the total number of matching articles, the number of pages, the current page, and a list of article results. By default it returns 10 results per page - we increased this to 50 using the page-size parameter. We restricted our search to articles published between 1 January 2010 and 31 December 2024, giving a 15-year window that covers several major UK events including the 2010 tuition fee protests, Brexit (2016), and COVID-19 (2020).

## 2.2 Fields Collected 
We collected the following fields for each article: id, webPublicationDate, sectionId, sectionName, webTitle, and wordcount. The wordcount field is not returned by default and had to be requested using the show-fields parameter. We also added a university column to track which search query each article came from. 

We also collected bodyText during the original data collection run, but excluded it from the saved CSV because the file would exceed GitHub's 100MB limit. The full data including body text can be reproduced by running data_collection.ipynb. 

## 2.3 Querying the API

In [6]:
#example only - do not run
import requests
import json

#load API key from keys.json
with open('keys.json') as f:
    keys = json.load(f)
guardian_api_key = keys['guardian']['api_key']

#single request for University College London articles, page 1
r = requests.get(
    'https://content.guardianapis.com/search',
    params={
        'q': '"University College London"',
        'api-key': guardian_api_key,
        'order-by': 'oldest',
        'from-date': '2010-01-01',
        'to-date': '2024-12-31',
        'page-size': 50,
        'page': 1,
        'show-fields': 'wordcount,bodyText'
    }
)

print('Status code:', r.status_code)
response = r.json()['response']
print('Total articles:', response['total'])
print('Total pages:', response['pages'])

Status code: 200
Total articles: 5227
Total pages: 105


## 2.4 Pagination 

Because each university has hundreds or thousands of articles, we could not get them all in one request. We used pagination to loop through all pages, incrementing the page parameter on each iteration and stopping when currentPage reached pages.

We also handled the rate limit (HTTP status 429) by pausing for 60 seconds and retrying, as demonstrated in the lecture. The full loop for all 10 universities is in data_collection.ipynb. A simplified version is shown below for illustration:

In [ ]:
#simplified pagination loop (not run here - see data_collection.ipynb for the full version)
import time

results = []
page = 1

#paginate through all pages of results
while True:
    try:
        r = requests.get(
            'https://content.guardianapis.com/search',
            params={
                'q': '"University College London"',
                'api-key': guardian_api_key,
                'order-by': 'oldest',
                'page': page,
                'page-size': 50,
                'from-date': '2010-01-01',
                'to-date': '2024-12-31',
                'show-fields': 'wordcount,bodyText'
            }
        )

        #successful response
        if r.status_code == 200:
            response = r.json()['response']
            results.append(response)
            print(f'Collected data from page {page}.')
            #stop if we've reached the last page
            if response['currentPage'] >= response['pages']:
                break
            time.sleep(1)  #wait 1 second between requests
            page += 1

        #rate limit hit - wait and retry the same page
        elif r.status_code == 429:
            print('Rate limit reached, waiting for 60 seconds')
            time.sleep(60)

        #something else went wrong
        else:
            print(f'Unexpected status code {r.status_code} on page {page}. Stopping.')
            break

    except Exception as e:
        print(f'Failed to collect data for page {page}. Error: {e}')
        break

## 2.5 Data Storage 

After collecting all articles for all 10 universities, we combined them into a single DataFrame and saved it to: 
data/raw/combined_raw.csv 

This file has 25,142 rows and 7 columns: university, id, webPublicationDate, sectionId, sectionName, webTitle, and wordcount. The bodyText column was not saved due to the Github file size limit but can be reproduced by re-running data_collection.ipynb. 

The full data collection process, including the complete pagination loop for all 10 universities and the code to flatten and save the JSON responses, is documented in data_collection.ipynb. 

To reproduce the data collection: save your Guardian API key in a file called keys.json in the project root directory, formatted as {"guardian": {"api_key": "YOUR_KEY"}}. Then run data_collection.ipynb. Note that the full collection takes approximately 30 minutes due to rate limiting.